# 8 · Measurement validity  `[EVAL]`

**Is the ruler trustworthy?** Notebooks 1–4 and 7 ask what the arms did; this one asks whether the
instrument that says so can be believed. Two questions, both read from the `data/eval_scores/` lake
that the paid `notebooks/scoring/Judge_Reliability.ipynb` writes into — this notebook only *reads*,
so it costs nothing and renders inside `render_views.py`.

- **§1 · Judge reliability** — the oracle's own repeatability (ICC), plus a decoupled second judge:
  do the endpoint contrasts keep their sign under a grader from a different model family that never
  played the patient?
- **§2 · Multi-judge** — where the variance in an arm mean actually comes from, whether gains
  transfer to a held-out grader, and at what effect size the two judges start agreeing.

> **Judge-invariant — which is why it is its own family.** Every artifact here contains *both*
> graders, so `reliability.py` loads them explicitly and ignores `EDA_JUDGE`. Exports go to
> `results/<VIEW>/figures|tables/8_measurement/` with **no `<judge>/` level**: a path naming one
> grader would assert that grader produced a cross-judge figure. `render_views.py` renders this
> notebook exactly once per view rather than once per grader.
>
> Split out of `5_Training_and_Reliability` on 2026-07-29 — that notebook is training-side and
> refuses a second judge, which forced these eval-side, cross-judge artifacts to be written under
> the primary oracle's folder.


In [ ]:
import sys, os
_p = os.path.abspath(".")                      # find eda/ (the dir holding eda_analysis/) from any depth
while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "eda_analysis")):
    _p = os.path.dirname(_p)
sys.path.insert(0, _p)
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
pd.set_option("display.width", 185, "display.max_columns", 50)
import eda_analysis
from eda_analysis import stats, training, figures, plots

# ╔═══ VIEW — the one knob ════════════════════════════════════════════════════════╗
# "all" = every arm | "L0" = K=0 arms (PTO_LA0/GRPO_LA0) | "L5" = K=5 arms (thin, paused).
# Sets BOTH the arm filter AND the results root -> results/<VIEW>/figures|tables/<group>/.
# Edit the default for interactive use; render_views.py overrides it via the EDA_VIEW env var.
VIEW = os.environ.get("EDA_VIEW", "L0")

cfg = eda_analysis.EdaConfig(
    view=VIEW,                             # arm filter + results/<VIEW>/ root
    export_group="8_measurement",          # JUDGE-INVARIANT family -> no <judge>/ level (exports.JUDGE_INVARIANT_GROUPS)
    selection="all",
    focus_arms=None, focus_metric="Q1Q2",
)
S = eda_analysis.notebook_setup(cfg)
# NOTE: no EDA_JUDGE handling, deliberately. Every section reads EVERY judge from the score lake via
# reliability.py, so the active-judge knob would change nothing here; S.SCORES is used only to
# intersect the judged model states with the current VIEW.


## 1 · Judge reliability — oracle ICC + a decoupled second judge  `[EVAL]`

**Purpose.** Answer the two measurement-validity questions the rest of the EDA has to assume, on the anchor-model subset (base + endpoints + the GRPO peak × Q1/Q2/MICI × 96 convs). Scored by `Judge_Reliability.ipynb` (paid, run manually); this section only READS `data/eval_scores/`, so it stays free and inside `render_views.py`.

1. **Repeatability (LIMITATIONS §1).** The same oracle re-scoring the same conversations 3×, seeds differing and nothing else → **ICC(2,1)** + mean |Δ|. This is the instrument's own measurement error.
2. **Second judge (LIMITATIONS §2).** The simulated patient and the grading oracle are the same model (`gpt-4o-mini`), so the generator and evaluator are coupled. A different-family judge (Claude Haiku 4.5) that never played the patient breaks that coupling.

**Read.** Agreement is bounded by *both* raters' noise — compare `pearson_r` to the `ceiling` column, never to 1.0. The `bias` column is a LEVEL offset (a harsher judge marks everything down) and is irrelevant to the thesis, whose claims are all *contrasts*: the load-bearing panel is **contrast preservation** — if the PTO−GRPO endpoint gap keeps its sign under a decoupled judge, the headline is not an artifact of the shared patient/oracle model.

In [ ]:
from eda_analysis import reliability as rel

# AUTO-DISCOVERED from what the second judge has actually scored — not a hardcoded subset.
# This section began life on a 4-model x 3-metric anchor subset; once the full sweep lands it
# covers all 29 model states x 8 rubrics, and hardcoding would silently keep reporting the old
# corner of a grid that has since been completed. §2b still restricts the ALL-PAIRS table to the
# anchor states (C(29,2) x 8 = 3,248 contrasts is unreadable and a multiple-comparisons trap).
ANCHOR_MODELS = ["PTOExp3_LA0_Base", "PTOExp3_LA0_I10", "GRPOExp3_LA0_I8", "GRPOExp3_LA0_I10"]
if rel.available():
    _cov = rel.coverage_table(rel.load_judge_long(rel.second_judge_tags()[0]))
    JUDGE_METRICS = [m for m in eda_analysis.QUESTIONNAIRE_ORDER if m in set(_cov.metric)]
    JUDGE_MODELS = sorted(set(_cov.model))
else:
    JUDGE_METRICS, JUDGE_MODELS = [], []
_in_view = [m for m in JUDGE_MODELS if m in set(S.SCORES.model)]
JUDGE_MODELS = _in_view or JUDGE_MODELS          # respect the VIEW's arm filter
print(f"[judge] {len(JUDGE_MODELS)} model states x {len(JUDGE_METRICS)} metrics scored by the second judge")

if not rel.available():
    print("No re-scoring data on disk — run Judge_Reliability.ipynb first (writes data/eval_scores/judge=<tag>/rep=<r>/).")
elif not _in_view:
    print(f"VIEW={VIEW}: the re-scoring subset is all K=0 — section skipped.")
else:
    TAG = rel.second_judge_tags()[0]
    JUDGE_NAME = rel.judge_display(TAG)

    # ── 7a · repeatability of the primary oracle ──────────────────────────────
    REP = rel.repeatability()
    if not REP.empty:
        display(rel.repeatability_by_metric(REP))
        eda_analysis.save_table(rel.repeatability_by_metric(REP), "oracle_repeatability_by_metric",
                                caption="Oracle repeatability per metric: ICC(2,1) across 3 re-scorings of the same conversations (seeds differ only) + the mean per-conversation |delta| between reps. The citable 'oracle noise' figure.")
        eda_analysis.save_table(REP, "oracle_repeatability_icc",
                                caption="Oracle repeatability per (metric, model): ICC(2,1) + mean |delta| across 3 re-scorings.")
        fig = plots.oracle_repeatability_bars(REP, metrics=JUDGE_METRICS)
        if fig:
            eda_analysis.save_fig(fig, "oracle_repeatability_icc",
                                  caption="ICC(2,1) per model and metric from 3 re-scorings of the same conversations; dotted = Koo & Li good (0.75) / excellent (0.90).")
            plt.show()

    # ── 7b · second judge vs the primary oracle ───────────────────────────────
    JL = rel.load_judge_long(TAG, reps=[0])
    PL = rel.load_primary_long(JUDGE_MODELS, JUDGE_METRICS)
    AGR = rel.agreement(JL, PL, REP)
    display(AGR)
    eda_analysis.save_table(AGR, "second_judge_agreement",
                            caption=f"Per-conversation agreement between {JUDGE_NAME} and the primary oracle (gpt-4o-mini): Pearson r, Spearman rho, level bias, and the attenuation ceiling implied by the primary oracle's ICC.")
    fig = plots.judge_agreement_scatter(JL, PL, agr_tab=AGR, metrics=JUDGE_METRICS,
                                        judge_name=JUDGE_NAME)
    if fig:
        eda_analysis.save_fig(fig, "judge_agreement_scatter",
                              caption=f"Per-conversation scores, {JUDGE_NAME} vs the primary oracle, one panel per metric. Distance from the dashed identity line is level bias; scatter around a trend is rank disagreement.")
        plt.show()

    display(rel.arm_means_by_judge(JL, PL, JUDGE_NAME))

    # ── 7c · THE defense check: does the contrast survive the judge swap? ──────
    CON = rel.contrasts(JL, PL, JUDGE_METRICS)
    display(CON)
    eda_analysis.save_table(CON, "second_judge_contrasts",
                            caption=f"Contrast preservation: each endpoint contrast as a paired delta over the 96 matched personas under the primary oracle and under {JUDGE_NAME}, with same_sign. The defense against the patient=oracle coupling in LIMITATIONS section 2.")
    fig = plots.judge_contrast_bars(CON, metrics=JUDGE_METRICS, judge_name=JUDGE_NAME)
    if fig:
        eda_analysis.save_fig(fig, "judge_contrast_preservation",
                              caption=f"Endpoint contrasts under both judges. Same-sign bars mean the result does not depend on the grader also having played the patient ({JUDGE_NAME} is a different model family).")
        plt.show()

    print("\nVERDICT:", rel.summary_line(REP, AGR, CON))

## 2 · Multi-judge — variance sources, transfer, and resolution  `[EVAL]`

**Purpose.** §1 asked *does the contrast survive a second judge?* (yes, on two hand-picked pairs). This section asks the three follow-ups a defence actually turns on. Free — reads the same `data/eval_scores/` lake.

**8a · Both judges, side by side.** A dumbbell per model state: bar *length* is the level offset, bar *order* is the claim. The two are never averaged — see the note above.

**8b · All pairwise contrasts.** *(anchor states only — all pairs of 29 states would be 3,248 contrasts: unreadable, and a multiple-comparisons trap.)* `DEFAULT_CONTRAST_PAIRS` checks two of the six pairs available among the four anchors. The two it never checked are the two the thesis leans on hardest: the **best-vs-best steelman** (PTO@10 vs GRPO@8 — the tightest margin in the chapter) and the **regression claim** (GRPO@8 vs GRPO@10 — the sycophancy mechanism). Pairing is on the recovered `persona_id`, not `file_index`: the trainer reshuffles the 96 personas every iteration, so a `file_index` join across unmatched iterations pairs unrelated conversations. Means are unaffected by that, but `dz` and the CI are not — and those are what a thesis table reports.

**8c · Variance decomposition.** Two-way random effects over arms × judges, on the arm means the thesis actually reports. Three components: **arm** (signal), **judge level** (large, and harmless — it cancels in every contrast), and **arm × judge** (the only one that threatens a claim: an ordering that depends on who is grading). `dependability_k1` is the generalizability coefficient for an arm mean read off a single judge — the number to quote when asked how far one judge's ranking can be trusted; `k2` is the same with both judges averaged, which is the honest answer to *"would a second judge help?"*.

**8d · Gain retention — the reward-hacking test.** What fraction of each arm's gain over Base survives the judge swap. Because the primary judge *was* the training reward and the second judge is held out, `Δ(judge) / Δ(primary)` is a **train/test generalization ratio**, not a reliability statistic: ~1.0 means the gain is a real behaviour change both judges see; ~0 means it lived only in the grader that was optimized. Uniform retention across arms is scale compression and uninteresting — the signal is retention that *differs by arm on one metric while staying flat on another*.

Once the full grid is scored, retention also becomes a **trajectory**: reward hacking is a process, so the sharper question is not *"did this endpoint transfer?"* but ***"at which iteration did the gains stop transferring?"*** — a line declining with training estimates when the policy began fitting its grader, which no single-endpoint comparison can give you.

**8e · Concordance vs effect size.** *"When the primary judge reports a gap of at least x, how often does the second judge agree on the direction?"* — a curve, not a scalar, because a single r is dominated by the 1.2–1.7 point level offset that cancels in every contrast, while a rank statistic discards the magnitude that decides whether a gap matters. ⚠ **Each point is a pair of single conversations**; the thesis compares 96-conversation means, which resolve ~10× better. Do not read a bin height as confidence in an arm-level claim — that is what 8a and `dependability_k1` are for. Exact primary-judge ties are excluded (they state no ordering to reproduce; counting them as failures pushes the smallest bin below chance).

In [ ]:
if not rel.available() or not _in_view:
    print("Multi-judge section skipped (no second-judge scores in this view).")
else:
    # The "gain over what?" baseline for 8d — derived from THIS view's judged frame, never
    # hardcoded: the L0/all views hold PTOExp3_LA0_Base, but the L5 view filters to K=5 arms,
    # where only the *_LA5_Base draws exist. A hardcoded LA0 name there made gain_retention()
    # skip every metric and save an EMPTY multijudge_gain_retention.md (caught 2026-08-18).
    # Prefer the PTO base to match the long-standing L0 convention (one shared reference draw).
    _bases = sorted(m for m in set(JL.model) & set(PL.model) if m.endswith("_Base"))
    _bases = [m for m in _bases if m.startswith("PTOExp3")] + _bases
    if not _bases:
        raise SystemExit("no *_Base model in this view's judged frame — cannot anchor gain retention")
    REFERENCE_MODEL = _bases[0]
    print(f"[retention] reference base = {REFERENCE_MODEL}")
    JUDGE_N_EXPECTED = 96                  # conversations per (metric, model) in a complete sweep

    # A second-judge sweep can land PARTIALLY (rate limits, expired batch, exhausted credit).
    # Partial cells are unbiased but less precise, and persona-paired stats collapse across two
    # partial arms — so restrict every table below to fully-scored cells and say what was dropped.
    COV = rel.coverage_table(JL, n_expected=JUDGE_N_EXPECTED)
    eda_analysis.save_table(COV, "multijudge_coverage",
                            caption=f"Conversations scored by {JUDGE_NAME} per (metric, model), out of {JUDGE_N_EXPECTED}. Section 8 analyses only cells marked complete; partial cells are reported here so a truncated sweep is never mistaken for full coverage.")
    if not COV.complete.all():
        print(f"[coverage] second-judge sweep is INCOMPLETE — "
              f"{int(COV.complete.sum())}/{len(COV)} cells fully scored "
              f"({COV.pct.min():.0f}-{COV.pct.max():.0f}% per cell). "
              f"Section 8 falls back to the complete cells only.")
    JL, PL = rel.filter_complete_cells(JL, PL, n_required=JUDGE_N_EXPECTED)
    JUDGE_METRICS = [m for m in JUDGE_METRICS if m in set(JL.metric)]
    if JL.empty or not JUDGE_METRICS:
        raise SystemExit("no fully-scored second-judge cells — nothing to analyse in section 2")

    # ── 8a · both judges side by side (never averaged) ────────────────────────
    fig = plots.judge_dumbbell(JL, PL, metrics=JUDGE_METRICS, judge_name=JUDGE_NAME)
    if fig:
        eda_analysis.save_fig(fig, "multijudge_arm_means_dumbbell",
                              caption=f"Arm means under the primary oracle and {JUDGE_NAME}. Bar LENGTH is the level offset (large, and it cancels in every contrast); bar ORDER is what the thesis claims. Deliberately not averaged: the primary judge was the training reward and the second is held out, and the offset is model-dependent.")
        plt.show()

    # ── 8b · every pairwise contrast, persona-paired ──────────────────────────
    # Every pair of 29 model states would be C(29,2) x 8 = 3,248 contrasts: unreadable as a table
    # and a multiple-comparisons trap. Enumerate all pairs among the ANCHOR states (base, both
    # endpoints, the GRPO peak) - the six that carry claims - and let 8d/8e use the full grid.
    CONTRAST_MODELS = [m for m in JUDGE_MODELS if m in set(JL.model)]
    PAIRS = rel.all_pairs_contrasts(JL, PL, JUDGE_METRICS, models=CONTRAST_MODELS)
    print(f"[contrasts] {len(CONTRAST_MODELS)} anchor states -> {len(PAIRS)} contrasts "
          f"({len(set(JL.model))} states scored in total)")
    display(PAIRS[["metric", "contrast", "primary_delta", "judge_delta",
                   "judge_ci_lo", "judge_ci_hi", "judge_dz", "same_sign"]])
    eda_analysis.save_table(PAIRS, "multijudge_all_pairs_contrasts",
                            caption=f"Every model pair x metric under both judges, paired on the recovered persona. judge_ci_* is a percentile bootstrap over personas. same_sign is the defence: {int(PAIRS.same_sign.sum())}/{len(PAIRS)} contrasts keep their direction under {JUDGE_NAME}, which never played the patient.")

    # The rate over that table is what the thesis quotes, and it is only interpretable against an
    # effect size: a pooled "88% agree" reads as weak until you see the disagreements sit entirely
    # in gaps too small to claim. Save the ladder so the narrative docs cite a tracked artifact.
    SIGN = rel.sign_preservation(PAIRS)
    SIGN_BY_METRIC = rel.sign_preservation(PAIRS, by=["metric"])
    display(SIGN)
    eda_analysis.save_table(SIGN, "multijudge_sign_preservation",
                            caption=f"Share of the {len(PAIRS)} pairwise arm x metric contrasts whose direction survives the swap to {JUDGE_NAME}, as a function of the gap the primary judge reports. Read the row at the effect size you are claiming; the pooled row is dragged down by contrasts too small to claim in the first place.")
    eda_analysis.save_table(SIGN_BY_METRIC, "multijudge_sign_preservation_by_metric",
                            caption="The same ladder per rubric. A rubric that preserves sign less often is one whose arm ordering depends on who is grading - compare against dependability_k1 in multijudge_variance_components, which measures the same weakness from a completely different direction. WARNING: the thresholds are ABSOLUTE, so a row is comparable to other rows of the SAME rubric, never across rubrics - PCT and MICI live on a 0-1 scale and never reach 0.25, while Q1/Q2/WAI-SR/MITI are 1-5 or 1-7. The cross-rubric comparison is the all-contrasts row.")

    # ── 8c · where does arm-mean variance come from? ──────────────────────────
    VC_CONV = rel.variance_components_conversation(JL, PL, JUDGE_METRICS)
    VC_ARM = rel.variance_components_arm(JL, PL, JUDGE_METRICS, conv_components=VC_CONV)
    display(VC_ARM)
    eda_analysis.save_table(VC_ARM, "multijudge_variance_components",
                            caption="Two-way random-effects decomposition of the arm means the thesis reports: arm (signal) vs judge level (cancels in contrasts) vs arm x judge (ordering that depends on the grader). dependability_k1/k2 = generalizability of an arm mean read off one judge vs both averaged.")
    eda_analysis.save_table(VC_CONV, "multijudge_variance_components_per_conversation",
                            caption="The same decomposition at the per-conversation level, per (metric, model). var_resid here is per-conversation judge disagreement, which is what attenuates cross-judge correlations.")
    fig = plots.variance_decomposition_bars(VC_ARM, metrics=JUDGE_METRICS)
    if fig:
        eda_analysis.save_fig(fig, "multijudge_variance_decomposition",
                              caption="Share of arm-mean variance by source. A large judge-level slice is harmless (it cancels in contrasts); the arm x judge slice is the only component that threatens a claim.")
        plt.show()

    # ── 8d · does the improvement transfer to a held-out judge? ───────────────
    RET = rel.gain_retention(JL, PL, REFERENCE_MODEL, JUDGE_METRICS)
    display(RET)
    eda_analysis.save_table(RET, "multijudge_gain_retention",
                            caption=f"Fraction of each arm's gain over {REFERENCE_MODEL} that survives the swap to {JUDGE_NAME}. The primary judge was the training reward and the second judge is held out, so this is a train/test generalization ratio: ~1.0 = a real behaviour change; ~0 = a gain that existed only in the optimized grader. CI is a persona bootstrap.")
    fig = plots.gain_retention_bars(RET, metrics=JUDGE_METRICS, judge_name=JUDGE_NAME)
    if fig:
        eda_analysis.save_fig(fig, "multijudge_gain_retention",
                              caption="Gain retention under a held-out judge, with persona-bootstrap CIs. Uniform bars across arms = scale compression; one arm collapsing while others hold = that arm's gain did not transfer.")
        plt.show()

    # Retention as a TRAJECTORY: with every iteration scored by both judges, "when did the gains
    # stop transferring?" becomes answerable, which no single-endpoint comparison can do.
    fig = plots.retention_trajectory(RET, metrics=JUDGE_METRICS, judge_name=JUDGE_NAME,
                                     palette=S.PALETTE)
    if fig:
        eda_analysis.save_fig(fig, "multijudge_retention_trajectory",
                              caption=f"Gain retention vs iteration under {JUDGE_NAME}. A line near 1.0 = gains a held-out judge also sees; a line declining with training = a policy progressively fitting the grader it was trained against, with the turn point estimating when that set in.")
        plt.show()

    # Q1-only single panel at column width. The retention claim in the reward-hacking draft rests
    # on the Q1 panel, and the full multi-metric grid is illegible at \columnwidth — this is the
    # figure the paper actually embeds (requested by the draft's Figure-3 legibility TODO).
    if "Q1" in set(RET.metric):
        fig = plots.retention_trajectory(RET, metrics=["Q1"], ncols=1, judge_name=JUDGE_NAME,
                                         palette=S.PALETTE)
        if fig:
            eda_analysis.save_fig(fig, "multijudge_retention_trajectory_Q1",
                                  caption=f"Gain retention vs iteration under {JUDGE_NAME}, Q1 only — the single panel the retention claim rests on, sized for a one-column figure. Same data as the Q1 panel of multijudge_retention_trajectory.")
            plt.show()

    # ── 8e · how much resolution does a gap of a given size carry? ────────────
    CONC = pd.concat([rel.concordance_by_effect_size(JL, PL, m, scope=s)
                      for m in JUDGE_METRICS for s in ("cross_model", "within_model")],
                     ignore_index=True)
    if not CONC.empty:
        display(CONC.pivot_table(index="bin", columns=["metric", "scope"], values="concordance"))
        eda_analysis.save_table(CONC, "multijudge_concordance_by_effect_size",
                                caption="P(second judge agrees on the direction) as a function of the gap the primary judge reports, per conversation PAIR. Exact primary-judge ties excluded. Not a confidence in any arm-level claim - arm means over 96 conversations resolve far better; see multijudge_all_pairs_contrasts.")
        fig = plots.concordance_curve(CONC, judge_name=JUDGE_NAME)
        if fig:
            eda_analysis.save_fig(fig, "multijudge_concordance_curve",
                                  caption="Cross-judge ordering agreement vs effect size, per conversation pair. Shows how much per-conversation resolving power a given gap carries - i.e. why 96 conversations per arm are needed.")
            plt.show()

    print("\nVERDICT:", rel.multi_judge_summary_line(VC_ARM, RET, PAIRS))

## 3 · How to read this notebook
- **ICC (§1)** is how much of a per-conversation score is signal rather than re-scoring noise. Read it against Koo & Li (2016): ≥0.75 good, ≥0.90 excellent. It bounds everything downstream — an arm difference smaller than the grader's own noise is not a difference.
- **Cross-judge `r` must be compared to the `ceiling`, never to 1.0.** The ceiling is `sqrt(ICC_primary × ICC_judge)`: two imperfect raters cannot correlate perfectly even when measuring the same thing. Both terms have been measured since 2026-07-28; `ceiling_basis` records whether a cell used measured values or fell back to the old `ICC_judge == ICC_primary` assumption.
- **`same_sign` is the load-bearing number**, not the correlation. A large level `bias` between judges is expected and harmless — the thesis reports contrasts, which cancel it. What would hurt is an arm *ordering* that depends on who grades.
- **Never average the two judges' raw scores.** The primary oracle *was the training reward*; the second judge never touched training. That is optimization-target vs held-out-test, not two interchangeable raters — `reliability.py` enforces this and only ever combines contrasts or standardized quantities.
- **`arm × judge` (§2) is the only variance component that can invalidate a claim.** A large `judge` term is a level shift; a large interaction means the ranking itself moves with the grader. Read `dependability_k1` as "how far can I trust an arm ranking taken off ONE judge".
- **Gain retention (§2) is the reward-hacking test**: `Δ(held-out) / Δ(trained-against)`. ~1.0 = a real behaviour change both graders see; ~0 = a gain that existed only in the optimized grader.
- **Concordance (§2) is per conversation PAIR, not per arm** — arm means over 96 conversations resolve ~10× better, so do not read a bin height as confidence in an arm-level claim.
- _(The measured values are narrated in `results/<view>/SUMMARY.md` §7 and the caveats they imply in `docs/LIMITATIONS.md` §1–§3. This notebook is where they are computed.)_


In [ ]:
print("index ->", eda_analysis.build_index())